In [ ]:
# Part 1 — Object Detection

from ultralytics import YOLO

# Load the YOLO detection model
model = YOLO("yolo26n.pt")

# Run inference on our image
results = model("bus.jpg")

#was Display the first result with:


# Also print the detection information
print("Detection completed!")
print("Number of detected objects:", len(results[0].boxes))
print("\nDetected objects:")


# Extract bounding-box coordinates

boxes = result.boxes.xyxy   # [N, 4] tensor
conf = result.boxes.conf    # [N] tensor
cls = result.boxes.cls      # [N] tensor


# Extracting the first detection
x1, y1, x2, y2 = boxes[0].tolist()

print("\nFirst Detection Coordinates:")
print("x1 =", x1)
print("y1 =", y1)
print("x2 =", x2)
print("y2 =", y2)

segmentation_model = YOLO("yolo26n-seg.pt")

segmentation_results = segmentation_model("bus.jpg")

segmentation_result = segmentation_results[0]
if segmentation_result.masks is not None:

    print(
        "Number of segmentation masks:",
        len(segmentation_result.masks)
    )

    print(
        "Mask shape:",
        segmentation_result.masks.data.shape
    )

    # Display the segmentation result.
    segmentation_result.show()



# ------------------------------------------------------------
# PART 1 COMPLETE
# ------------------------------------------------------------

print("\n=== PART 1 COMPLETED ===")

In [ ]:
# PART 2 — OBJECT TRACKING

# YOLO tracking detects and tracks objects across video frames.
# Each tracked object receives an ID.


from ultralytics import YOLO
import cv2

# Load the YOLO model
model = YOLO("yolo26n.pt")

# Open the traffic video
video = cv2.VideoCapture("road_trafifc.mp4")

print("=== TRAFFIC TRACKING ===")

while video.isOpened():

    # Read one frame
    success, frame = video.read()

    if not success:
        break

    # Run YOLO tracking
    results = model.track( frame, persist=True
    )

    # Draw bounding boxes and tracking IDs
    annotated_frame = results[0].plot()

    # Display the frame
    cv2.imshow( "Traffic Object Tracking",  annotated_frame
    )

    # Press Q to stop
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# Close everything
video.release()
cv2.destroyAllWindows()

print("Tracking completed!")
print("=== PART 2 COMPLETED ===")

In [ ]:
# Part 3 — Model Evaluation

from ultralytics import YOLO

# Load the YOLO model
model = YOLO("yolo26n.pt")

# Run validation
results = model.val(data="coco8.yaml")

# ------------------------------------------------------------
# Overall evaluation metrics
# ------------------------------------------------------------

print("=== OVERALL RESULTS ===")

print("Precision:", results.box.mp)
print("Recall:", results.box.mr)
print("mAP50:", results.box.map50)
print("mAP50-95:", results.box.map)


# ------------------------------------------------------------
# Per-class results
# ------------------------------------------------------------

print("\n=== PER-CLASS RESULTS ===")

for i, class_name in enumerate(model.names.values()):

    precision = results.box.p[i]
    recall = results.box.r[i]
    map50 = results.box.ap50[i]
    map50_95 = results.box.ap[i].mean()

    print(
        f"{class_name}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"mAP50={map50:.3f}, "
        f"mAP50-95={map50_95:.3f}"
    )


# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\n=== INTERPRETATION ===")

print(
    "Precision shows how many predicted objects were correct."
)

print(
    "Recall shows how many actual objects were detected."
)

print(
    "mAP50 evaluates detection performance at IoU = 0.50."
)

print(
    "mAP50-95 evaluates performance across stricter IoU thresholds."
)

print(
    "Low recall can indicate missed objects (false negatives)."
)

print(
    "Low precision can indicate incorrect detections (false positives)."
)

print(
    "The validation results can be used to identify which classes "
    "need further improvement."
)

print("\n=== PART 3 COMPLETED ===")

In [ ]:
# Part 4.1 —  Dataset Preparation

import os
import random
import shutil

# Original dataset folder
source = "training_image"

# New YOLO dataset folders
dataset = "vehicle_dataset"

train_images = os.path.join(dataset, "images", "train")
val_images = os.path.join(dataset, "images", "val")

train_labels = os.path.join(dataset, "labels", "train")
val_labels = os.path.join(dataset, "labels", "val")

# Create the folders
for folder in [train_images, val_images, train_labels, val_labels]:
    os.makedirs(folder, exist_ok=True)

# Find all images that have a matching label file
files = []

for file in os.listdir(source):

    if file.lower().endswith(".jpg"):

        image = os.path.join(source, file)
        label = os.path.join(source, os.path.splitext(file)[0] + ".txt")

        if os.path.exists(label):
            files.append((image, label))

# Shuffle the dataset
random.seed(42)
random.shuffle(files)

# Use 80% for training and 20% for validation
split = int(len(files) * 0.8)

train_files = files[:split]
val_files = files[split:]

# Copy training images and labels
for image, label in train_files:
    shutil.copy(image, train_images)
    shutil.copy(label, train_labels)

# Copy validation images and labels
for image, label in val_files:
    shutil.copy(image, val_images)
    shutil.copy(label, val_labels)

# Create the YOLO dataset configuration
with open("vehicle.yaml", "w") as f:

    f.write("path: vehicle_dataset\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n\n")

    f.write("names:\n")
    f.write("  0: Ambulance\n")
    f.write("  1: Bus\n")
    f.write("  2: Car\n")
    f.write("  3: Truck\n")

# Print the results
print("=== DATASET PREPARED ===")
print("Total labeled images:", len(files))
print("Training images:", len(train_files))
print("Validation images:", len(val_files))
print("Classes: Ambulance, Bus, Car, Truck")
print("Created: vehicle.yaml")

print("\n=== READY FOR TRAINING ===")

=== DATASET PREPARED ===
Total labeled images: 390
Training images: 312
Validation images: 78
Classes: Ambulance, Bus, Car, Truck
Created: vehicle.yaml

=== READY FOR TRAINING ===


In [3]:
# Part 4.2 — Custom Model Training

from ultralytics import YOLO

# Load a pretrained YOLO model
model = YOLO("yolo26n.pt")

# Train the model on our custom vehicle dataset
results = model.train(
    data="vehicle.yaml",
    epochs=10,
    imgsz=640,
    batch=8
)

# Print training information
print("\n=== CUSTOM TRAINING COMPLETED ===")

print("Dataset: Custom Vehicle Dataset")
print("Training images: 312")
print("Validation images: 78")
print("Classes: Ambulance, Bus, Car, Truck")
print("Epochs: 10")
print("Image size: 640")

print("\nTraining was completed using model.train().")
print("The training results were saved by Ultralytics.")

print("\n=== PART 4 COMPLETED ===")

Ultralytics 8.4.130  Python-3.14.7 torch-2.13.0+cpu CPU (Intel Core 7 150U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=vehicle.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, op